In [0]:
df_cal = spark.read.format('csv')\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("abfss://bronze@deprojectdatalakenith1.dfs.core.windows.net/AdventureWorks_Calendar")

display(df_cal)

In [0]:
df_cus = spark.read.format('csv')\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("abfss://bronze@deprojectdatalakenith1.dfs.core.windows.net/AdventureWorks_Customers")

display(df_cus)

In [0]:
df_procat = spark.read.format('csv')\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("abfss://bronze@deprojectdatalakenith1.dfs.core.windows.net/AdventureWorks_Product_Categories")

display(df_procat)

In [0]:
df_pro = spark.read.format('csv')\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("abfss://bronze@deprojectdatalakenith1.dfs.core.windows.net/AdventureWorks_Products")

display(df_pro)

In [0]:
df_ret = spark.read.format('csv')\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("abfss://bronze@deprojectdatalakenith1.dfs.core.windows.net/AdventureWorks_Returns")

display(df_ret)

In [0]:
df_sales = spark.read.format('csv')\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("abfss://bronze@deprojectdatalakenith1.dfs.core.windows.net/AdventureWorks_Sales*")

display(df_sales)

In [0]:
df_ter = spark.read.format('csv')\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("abfss://bronze@deprojectdatalakenith1.dfs.core.windows.net/AdventureWorks_Territories")

display(df_ter)

In [0]:
df_subcat = spark.read.format('csv')\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("abfss://bronze@deprojectdatalakenith1.dfs.core.windows.net/Product_Subcategories")

display(df_subcat)

transformations



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *


Calendar


In [0]:
df_cal = df_cal.withColumn("Month", month(df_cal.Date)).withColumn("Year", year(df_cal.Date))
df_cal.write.format("parquet").mode("append").option("path", "abfss://silver@deprojectdatalakenith1.dfs.core.windows.net/AdventureWorks_Calendar").save()



Customers


In [0]:
df_cus = df_cus.withColumn("FullName", concat_ws(" ",col("Prefix"), col("FirstName"), col("LastName")))
df_cus.write.format("parquet").mode("append").option("path", "abfss://silver@deprojectdatalakenith1.dfs.core.windows.net/AdventureWorks_Customers").save()


Product Categories


In [0]:
df_procat.write.format("parquet").mode("append").option("path", "abfss://silver@deprojectdatalakenith1.dfs.core.windows.net/AdventureWorks_Product_Categories").save()


Products

In [0]:
df_pro = df_pro.withColumn("ProductSKU",split(col("ProductSKU"),"-")[0])\
    .withColumn("ProductName",split(col("ProductName")," ")[0])

df_pro.write.format("parquet").mode("append").option("path", "abfss://silver@deprojectdatalakenith1.dfs.core.windows.net/AdventureWorks_Products").save()

returns


In [0]:
df_ret.write.format("parquet").mode("append").option("path", "abfss://silver@deprojectdatalakenith1.dfs.core.windows.net/AdventureWorks_Returns").save()

Territories

In [0]:
df_ret.write.format("parquet").mode("append").option("path", "abfss://silver@deprojectdatalakenith1.dfs.core.windows.net/AdventureWorks_Territories").save()

Product Sub Categories

In [0]:
df_subcat.write.format("parquet").mode("append").option("path", "abfss://silver@deprojectdatalakenith1.dfs.core.windows.net/Product_Subcategories").save()

Sales

In [0]:
df_sales = df_sales.withColumn("StockDate", to_timestamp(col("StockDate")))\
    .withColumn("OrderNumber", regexp_replace(col("OrderNumber"), 'S','T'))\
    .withColumn("multiply", col("OrderLineItem") * col("OrderQuantity"))

df_sales.write.format("parquet").mode("append").option("path", "abfss://silver@deprojectdatalakenith1.dfs.core.windows.net/Product_Sales").save()


In [0]:
df_sales.groupBy("OrderDate").agg(count("OrderNumber")).alias('total_order').display()

Databricks visualization. Run in Databricks to view.